### 5. Feature Engineering
**Mục tiêu cốt lõi:**
- Tạo ra các biến mới (như Cuối tuần, Nhóm tuổi) để mô hình bắt được các quy luật mua sắm phức tạp
- Loại bỏ các cột không mang thông tin (như invoice_no, customer_id) để giảm nhiễu
- Giảm chiều dữ liệu và chuẩn hóa thang đo giúp thuật toán hội tụ nhanh hơn
- Giúp mô hình hoạt động tốt trên tập Test và dữ liệu thực tế

Các bước thực hiện:
- Data split: Chia tập train/test theo tỷ lệ sample 80/20
- Feature Extraction:
    - Date features: Biến invoice_date chứa rất nhiều "kho báu" ẩn. Nhóm sẽ trích xuất thành các cột mới: Day, Month, Year và biến nhị phân Is_Weekend (1 nếu là Thứ 7/CN, 0 nếu là ngày thường). Insight: Hành vi chi tiêu cuối tuần thường cao hơn ngày thường
- Feature Transformation:
    - Log-Transform: Biến mục tiêu price (Doanh thu) bị lệch phải nghiêm trọng (Right-skewed). Nhóm áp dụng np.log1p() để kéo phân phối về hình chuông, giúp thỏa mãn giả định của Linear Regression.
    - Rời rạc hóa (Binning): Chuyển đổi cột age (tuổi từ 18-90) thành các nhóm (Bins) như: Gen Z (<25), Millennials (25-40), Gen X (41-60), Boomers (>60). Việc này giúp mô hình dễ dàng học được sự khác biệt về chi tiêu giữa các thế hệ thay vì coi tuổi tác là một đường thẳng tuyến tính
- Encoding Categorical Variables:
    - Gom tất cả các biến chữ (gồm biến gốc như gender, category, shopping_mall và biến mới tạo như Age_Group) vào bộ One-Hot Encoding (OHE) để bẻ thành ma trận nhị phân
- Scaling (Chuẩn hóa thang đo): Áp dụng Standardization (Z-score) cho các biến số liên tục (như quantity hoặc age nếu không dùng Binning) để kéo Mean = 0 và Std = 1. Việc này giúp thuật toán Gradient Descent trong Ridge Regression không bị chệch hướng bởi các biến có giá trị quá lớn

In [12]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
data = pd.read_csv('./data/preprocessed-data/preprocessed_customer_shopping_data.csv', parse_dates=['invoice_date'])

In [4]:
# Tách Feature (X) và Target (y)
X = data.drop(columns=['revenue'])
y = data['revenue']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")
print("\nCác cột features còn lại trong X_train:")
print(X_train.columns.tolist())

Kích thước X_train: (79448, 6)
Kích thước X_test: (19862, 6)

Các cột features còn lại trong X_train:
['gender', 'age', 'category', 'payment_method', 'invoice_date', 'shopping_mall']


In [6]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79448 entries, 96196 to 15795
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   gender          79448 non-null  object        
 1   age             79448 non-null  int64         
 2   category        79448 non-null  object        
 3   payment_method  79448 non-null  object        
 4   invoice_date    79448 non-null  datetime64[ns]
 5   shopping_mall   79448 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 4.2+ MB


In [ ]:
def extract_date_features(X_data):
    """Trích xuất đặc trưng thời gian"""
    X = X_data.copy()
    X['Day'] = X['invoice_date'].dt.day
    X['Month'] = X['invoice_date'].dt.month
    X['Year'] = X['invoice_date'].dt.year
    # Is_Weekend: Thứ 7 (5) và CN (6) thì gán là 1, ngày thường gán là 0
    X['Is_Weekend'] = X['invoice_date'].dt.dayofweek.isin([5, 6]).astype(int)
    # Xóa cột Datetime gốc vì mô hình không đọc được
    return X.drop(columns=['invoice_date'])

In [ ]:
def bin_age(X_data):
    """Rời rạc hóa (Binning) nhóm tuổi"""
    X = X_data.copy()
    bins = [0, 25, 40, 60, 100]
    labels = ['Gen Z', 'Millennials', 'Gen X', 'Boomers']
    X['Age_Group'] = pd.cut(X['age'], bins=bins, labels=labels)
    # Xóa cột age gốc để tránh đa cộng tuyến
    return X.drop(columns=['age'])

In [9]:
X_train = extract_date_features(X_train)
X_test = extract_date_features(X_test)

X_train = bin_age(X_train)
X_test = bin_age(X_test)

In [11]:
# Log-Transform cho biến mục tiêu y (Khắc phục phân phối lệch phải)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [13]:
# Day, Month, Year được coi là biến số (Numeric) -> Cần Scale
numeric_features = ['Day', 'Month', 'Year']

# Các biến chữ -> Cần One-Hot Encoding
categorical_features = ['gender', 'category', 'payment_method', 'shopping_mall', 'Age_Group']

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features) # sparse_output=False để trả về ma trận đặc
    ],
    remainder='passthrough' 
)

In [15]:
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

In [20]:
print(X_train_final)

[[ 0.05016324  1.36855612 -0.98775603 ...  0.          0.
   0.        ]
 [ 1.30059999  0.24823581 -0.98775603 ...  0.          0.
   0.        ]
 [ 0.84589572 -0.31192434  0.58388622 ...  0.          1.
   0.        ]
 ...
 [ 0.50486751 -0.03184427  0.58388622 ...  0.          0.
   0.        ]
 [ 0.84589572 -1.43224465 -0.98775603 ...  0.          0.
   1.        ]
 [ 1.07324785 -0.8720845  -0.98775603 ...  1.          0.
   0.        ]]


In [18]:
print(f"   - Tập X_train_final: {X_train_final.shape}")
print(f"   - Tập X_test_final: {X_test_final.shape}")

   - Tập X_train_final: (79448, 31)
   - Tập X_test_final: (19862, 31)


In [22]:
os.makedirs('./data/ready_for_train', exist_ok=True)

joblib.dump(X_train_final, './data/ready_for_train/X_train_final.pkl')
joblib.dump(X_test_final, './data/ready_for_train/X_test_final.pkl')

joblib.dump(y_train_log, './data/ready_for_train/y_train_log.pkl')
joblib.dump(y_test_log, './data/ready_for_train/y_test_log.pkl')

# Dành cho Model Deployment
joblib.dump(preprocessor, './data/ready_for_train/preprocessor.pkl')

['./data/ready_for_train/preprocessor.pkl']